In [1]:
DATA_DIR = "../data/dataquest"

!pip install -q pycaret
!pip install -q catboost
!pip install -q xgboost
!pip install -q lightgbm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.8/165.8 kB 8.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 486.1/486.1 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.8/106.8 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.8/21.8 MB 96.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.9/81.9 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.2/302.2 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 82.3 MB/s eta 0:00:00
   ━━

In [2]:
import pandas as pd
import numpy as np
import time
import pickle as pkl

from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score, cross_val_predict, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import mean_squared_error

from catboost import CatBoostRegressor, Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

In [30]:
import pycaret
pycaret.__version__

'3.3.2'

In [31]:
import sklearn
sklearn.__version__

'1.2.2'

In [3]:
import pandas as pd

In [4]:
ready_train_path = f"{DATA_DIR}/train_ready_2.csv"

In [5]:
ready_df = pd.read_csv(ready_train_path)

# Modelling

In [6]:
X, y = ready_df.drop(["rain_1h"], axis=1), ready_df["rain_1h"]

In [7]:
X.drop([
    'datetime',
    'feels',
    'wind_spd',
    'wind_deg',
], axis=1, inplace=True)

## helper

In [8]:
from sklearn.model_selection import StratifiedKFold, KFold
import numpy as np
import pickle as pkl
import time

def regression_oof_preds(
    regressor,
    model_key,
    X_train,
    y_train,
    # X_test,
    metric_score,
    metric_score_args={},
    n_folds=5,
    dir_name="output",
    verbose=False,
):
    # test_preds = np.zeros(len(X_test))
    oof_full = np.zeros(len(y_train))

    # Start timer
    start_time = time.time()

    # k-fold cross validation for regression
    cv = KFold(n_splits=n_folds, shuffle=True, random_state=0)

    score = 0
    for fold, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train)):
        # Get training and validation sets
        x_tr, x_valid = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_valid = y_train.iloc[train_idx], y_train.iloc[val_idx]

        # Train using training split
        regressor.fit(x_tr, y_tr)

        # Out-of-fold predictions
        oof_preds = regressor.predict(x_valid)
        score += metric_score(y_valid, oof_preds, **metric_score_args) / n_folds
        oof_full[val_idx] = oof_preds

        # Test set predictions
        # test_preds += regressor.predict(X_test) / n_folds

    # Stop timer
    stop_time = time.time()

    if verbose:
        # Print score and time
        print("Model:", model_key)
        print("Average Metric:", np.round(score, 4))
        print("Training time (mins):", np.round((stop_time - start_time) / 60, 2))
        print("")

    # Save oof and test set preds
    with open(f"{model_key}_oof_preds.pkl", "wb") as f:
        pkl.dump(oof_full, f)
    # with open(f"{dir_name}/{model_key}_test_preds.pkl", "wb") as f:
    #     pkl.dump(test_preds, f)

    # return score, oof_full, test_preds
    return score, oof_full


## Catboost

In [13]:
cb_model = CatBoostRegressor(
    task_type="GPU",
    loss_function="RMSE",
    verbose=200,
    random_state=42
)

In [14]:
n_folds = 5
dir_name = ""

# Train CatBoost model
cb_score, cb_oof = regression_oof_preds(
    cb_model,
    "CatBoost",
    X,
    y,
    # test_df,
    metric_score=mean_squared_error,
    # metric_score_args={"average": "weighted"},
    n_folds=n_folds,
    dir_name=dir_name,
    verbose=True,
)

Learning rate set to 0.079647
0:	learn: 0.8845099	total: 28ms	remaining: 28s
200:	learn: 0.6684363	total: 4.77s	remaining: 19s
400:	learn: 0.6432356	total: 6.34s	remaining: 9.47s
600:	learn: 0.6253772	total: 7.9s	remaining: 5.24s
800:	learn: 0.6106090	total: 9.45s	remaining: 2.35s
999:	learn: 0.5968947	total: 11s	remaining: 0us
Learning rate set to 0.079647
0:	learn: 0.8832187	total: 35.6ms	remaining: 35.6s
200:	learn: 0.6667498	total: 4.71s	remaining: 18.7s
400:	learn: 0.6429840	total: 6.27s	remaining: 9.36s
600:	learn: 0.6250610	total: 7.83s	remaining: 5.2s
800:	learn: 0.6098630	total: 9.37s	remaining: 2.33s
999:	learn: 0.5959012	total: 11s	remaining: 0us
Learning rate set to 0.079647
0:	learn: 0.8863555	total: 42.6ms	remaining: 42.6s
200:	learn: 0.6680898	total: 4.59s	remaining: 18.2s
400:	learn: 0.6440404	total: 6.13s	remaining: 9.16s
600:	learn: 0.6256847	total: 7.7s	remaining: 5.11s
800:	learn: 0.6103554	total: 9.27s	remaining: 2.3s
999:	learn: 0.5963862	total: 10.8s	remaining: 0

## LightGBM

In [19]:
lgbm_model = LGBMRegressor(
    # device="gpu",
    verbose=0,
    random_state=42
)

In [21]:
n_folds = 5
dir_name = ""

# Train LightGBM model
lgbm_score, lgbm_oof = regression_oof_preds(
    lgbm_model,
    "LightGBM",
    X,
    y,
    # test_df,
    metric_score=mean_squared_error,
    # metric_score_args={"average": "weighted"},
    n_folds=n_folds,
    dir_name=dir_name,
    verbose=True,
)

Model: LightGBM
Average Metric: 0.4616
Training time (mins): 2.75



## xgboost

In [22]:
xgb_model = XGBRegressor(
    tree_method="gpu_hist",
    random_state=42
)

In [23]:
n_folds = 5
dir_name = ""

# Train LightGBM model
cb_score, cb_oof = regression_oof_preds(
    xgb_model,
    "XGB",
    X,
    y,
    # test_df,
    metric_score=mean_squared_error,
    # metric_score_args={"average": "weighted"},
    n_folds=n_folds,
    dir_name=dir_name,
    verbose=True,
)

Model: XGB
Average Metric: 0.4668
Training time (mins): 0.92



# Hillclimb

In [ ]:
# Join oof preds
oof_df = pd.DataFrame(index=np.arange(len(y)))
for i in ["CatBoost", "LightGBM", "XGBoost"]:
    with open(f"output/{i}_oof_preds.pkl", "rb") as f:
        oof_preds = pkl.load(f)
    df = pd.DataFrame({i: oof_preds.tolist()})
    oof_df = pd.concat([oof_df, df], axis=1)  # axis = 1, add on another column

# Join test preds
# test_preds_df = pd.DataFrame(index=np.arange(len(test_df)))
# for i in ["CatBoost", "LightGBM", "XGBoost"]:
#     with open(f"output/{i}_test_preds.pkl", "rb") as f:
#         test_preds = pkl.load(f)
#     df = pd.DataFrame({i: test_preds.tolist()})
#     test_preds_df = pd.concat([test_preds_df, df], axis=1)

display(oof_df.head(3))
# display(test_preds_df.head(3))

In [ ]:
# Evaluate oof preds
scores = {}
for col in oof_df.columns:
    scores[col] = mean_squared_error(y, oof_df[col])

# Sort scores
scores = {
    k: v for k, v in sorted(scores.items(), key=lambda item: item[1], reverse=True)
}

# Sort oof_df and test_preds
oof_df = oof_df[list(scores.keys())]
# test_preds_df = test_preds_df[list(scores.keys())]

scores

In [ ]:
# Initialise
STOP = False
current_best_ensemble = oof_df.iloc[:, 0]
current_best_test_preds = test_preds_df.iloc[:, 0]
MODELS = oof_df.iloc[:, 1:]  # the second, third, ... best models
weight_range = np.arange(
    0.01, 0.51, 0.01
)  # or with negative weights: np.arange(-0.5,0.51,0.01)
history = [mean_squared_error(y, current_best_ensemble)]
i = 0

# Hill climbing
while not STOP:
    i += 1
    potential_new_best_cv_score = mean_squared_error(
        y, current_best_ensemble
    )
    k_best, wgt_best = None, None
    for k in MODELS:
        for wgt in weight_range:
            potential_ensemble = (1 - wgt) * np.array(
                current_best_ensemble.tolist()
            ) + wgt * np.array(MODELS[k].tolist())
            cv_score = mean_squared_error(y, potential_ensemble)
            if cv_score < potential_new_best_cv_score:
                potential_new_best_cv_score = cv_score
                k_best, wgt_best = k, wgt

    if k_best is not None:
        current_best_ensemble = (1 - wgt_best) * np.array(
            current_best_ensemble.tolist()
        ) + wgt_best * np.array(MODELS[k_best].tolist())
        current_best_test_preds = (1 - wgt_best) * np.array(
            current_best_test_preds.tolist()
        ) + wgt_best * np.array(test_preds_df[k_best].tolist())
        MODELS.drop(
            k_best, axis=1, inplace=True
        )  # model kedua yang sudah ditambahkan, didrop, tidak dipertimbangkan lagi
        if MODELS.shape[1] == 0:
            STOP = True
        print(
            f"Iteration: {i}, Model added: {k_best}, Best weight: {wgt_best:.2f}, Best metric: {potential_new_best_cv_score:.5f}"
        )
        history.append(potential_new_best_cv_score)
    else:
        STOP = True